# Model 1 — Linear Regression

This notebook trains **Model 1: Linear Regression** using the train/test datasets created by `modeling_preparation_baseline.ipynb`.

### Goals
- Use the same chronological train/test split as the naive baseline.
- Treat `PULocationID` and calendar fields as categorical variables.
- Use historical demand, rolling demand, and weather as numerical predictors.
- Evaluate using **MAE, RMSE and R²**.
- Compare Model 1 against the **168-hour historical baseline**.
- Save the trained preprocessing + regression pipeline, predictions, metrics, and coefficients.

**Important:** This is Review 1 supervised demand prediction. The lag features are assumed to be available up to the prediction timestamp. This is different from Review 2 multi-step future forecasting.


## 1. Imports and configuration


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print('Libraries imported successfully.')


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Libraries imported successfully.


## 2. Locate the repository and load the prepared data


In [ ]:
def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'modeling' / 'train.csv').exists():
            return path
    raise FileNotFoundError(
        'Could not find data/modeling/train.csv. '
        'Run modeling_preparation_baseline.ipynb first.'
    )

REPO_ROOT = find_repo_root()
MODELING_DIR = REPO_ROOT / 'models' / 'modeling'

TRAIN_PATH = MODELING_DIR / 'train.csv'
TEST_PATH = MODELING_DIR / 'test.csv'
BASELINE_METRICS_PATH = MODELING_DIR / 'baseline_metrics.csv'

train_df = pd.read_csv(TRAIN_PATH, parse_dates=['pickup_hour'])
test_df = pd.read_csv(TEST_PATH, parse_dates=['pickup_hour'])

print('Repository:', REPO_ROOT)
print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
print('Train period:', train_df['pickup_hour'].min(), '->', train_df['pickup_hour'].max())
print('Test period :', test_df['pickup_hour'].min(), '->', test_df['pickup_hour'].max())


Repository: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting
Train shape: (460920, 17)
Test shape : (115368, 17)
Train period: 2025-09-08 00:00:00 -> 2026-01-25 03:00:00
Test period : 2026-01-25 04:00:00 -> 2026-02-28 23:00:00


## 3. Define target and features

### Target
`trip_count`

### Categorical predictors
- `PULocationID`
- `hour`
- `day_of_week`
- `month`

`is_weekend` is intentionally not used here because it is completely determined by `day_of_week`; including both in an ordinary linear model would introduce redundant information.

### Numerical predictors
- weather: apparent temperature, rain, snowfall, precipitation probability
- holiday indicator
- lag demand: 1h, 24h, 168h
- rolling demand: 3h, 24h

`pickup_hour` itself is used only for chronology and is not passed directly into the model.


In [3]:
TARGET = 'trip_count'

CATEGORICAL_FEATURES = [
    'PULocationID',
    'hour',
    'day_of_week',
    'month'
]

NUMERICAL_FEATURES = [
    'apparent_temperature',
    'rain',
    'snowfall',
    'precipitation_probability',
    'is_holiday',
    'trip_count_lag_1h',
    'trip_count_lag_24h',
    'trip_count_lag_168h',
    'trip_count_roll_3h',
    'trip_count_roll_24h'
]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

missing_required = [c for c in [TARGET] + FEATURES if c not in train_df.columns or c not in test_df.columns]
if missing_required:
    raise ValueError(f'Missing required modelling columns: {missing_required}')

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print('Number of features before encoding:', len(FEATURES))
print('Categorical features:', CATEGORICAL_FEATURES)
print('Numerical features:', NUMERICAL_FEATURES)


Number of features before encoding: 14
Categorical features: ['PULocationID', 'hour', 'day_of_week', 'month']
Numerical features: ['apparent_temperature', 'rain', 'snowfall', 'precipitation_probability', 'is_holiday', 'trip_count_lag_1h', 'trip_count_lag_24h', 'trip_count_lag_168h', 'trip_count_roll_3h', 'trip_count_roll_24h']


## 4. Validate the input data


In [4]:
for name, frame in [('X_train', X_train), ('X_test', X_test)]:
    missing = frame.isna().sum().sum()
    print(f'{name} missing values:', missing)

print('Negative target values in train:', int((y_train < 0).sum()))
print('Negative target values in test :', int((y_test < 0).sum()))

assert (y_train >= 0).all(), 'Training target contains negative values.'
assert (y_test >= 0).all(), 'Test target contains negative values.'


X_train missing values: 0
X_test missing values: 0
Negative target values in train: 0
Negative target values in test : 0


## 5. Build the Linear Regression pipeline

Categorical variables are one-hot encoded with the first category dropped as the reference category. Numerical variables use median imputation for robustness.

The model itself is ordinary `LinearRegression`; no hyperparameter tuning or regularization is being introduced at this stage.


In [5]:
# Compatibility with older and newer scikit-learn versions.
try:
    one_hot = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=True)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown='ignore', drop='first', sparse=True)

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', one_hot)
])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
    ('numerical', numerical_pipeline, NUMERICAL_FEATURES)
], remainder='drop')

model_1 = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

print(model_1)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['PULocationID', 'hour',
                                                   'day_of_week', 'month']),
                                                 ('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                    

## 6. Train Model 1


In [6]:
print('Training Linear Regression...')
model_1.fit(X_train, y_train)
print('Training completed.')


Training Linear Regression...
Training completed.


## 7. Generate predictions

Because taxi demand cannot be negative, the raw Linear Regression prediction is clipped at zero for the final demand prediction.


In [7]:
train_pred_raw = model_1.predict(X_train)
test_pred_raw = model_1.predict(X_test)

train_pred = np.clip(train_pred_raw, 0, None)
test_pred = np.clip(test_pred_raw, 0, None)

print('Raw test prediction range:', test_pred_raw.min(), 'to', test_pred_raw.max())
print('Clipped test prediction range:', test_pred.min(), 'to', test_pred.max())


Raw test prediction range: -32.31205028218852 to 809.9179811062214
Clipped test prediction range: 0.0 to 809.9179811062214


/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 8. Evaluate Model 1

The same metrics used for the baseline are used here so the comparison is fair.


In [8]:
def calculate_metrics(y_true, y_pred, dataset_name):
    return {
        'dataset': dataset_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
        'rows': len(y_true)
    }

model_1_metrics = pd.DataFrame([
    calculate_metrics(y_train, train_pred, 'train'),
    calculate_metrics(y_test, test_pred, 'test')
])

display(model_1_metrics)


,dataset,MAE,RMSE,R2,rows
0,train,7.305019,16.400304,0.955346,460920
1,test,7.486082,15.703616,0.947828,115368


## 9. Compare Model 1 with the naive baseline


In [9]:
if BASELINE_METRICS_PATH.exists():
    baseline_metrics = pd.read_csv(BASELINE_METRICS_PATH)
    baseline_row = baseline_metrics.iloc[0]

    comparison = pd.DataFrame([
        {
            'model': 'Naive 168-hour historical baseline',
            'MAE': baseline_row['MAE'],
            'RMSE': baseline_row['RMSE'],
            'R2': baseline_row['R2']
        },
        {
            'model': 'Model 1 - Linear Regression',
            'MAE': model_1_metrics.loc[model_1_metrics['dataset'] == 'test', 'MAE'].iloc[0],
            'RMSE': model_1_metrics.loc[model_1_metrics['dataset'] == 'test', 'RMSE'].iloc[0],
            'R2': model_1_metrics.loc[model_1_metrics['dataset'] == 'test', 'R2'].iloc[0]
        }
    ])

    display(comparison)
else:
    print('Baseline metrics file not found. Run the preparation notebook first.')


,model,MAE,RMSE,R2
0,Naive 168-hour historical baseline,9.307390,24.056583,0.877566
1,Model 1 - Linear Regression,7.486082,15.703616,0.947828


## 10. Save predictions, metrics and model


In [10]:
prediction_output = test_df[['PULocationID', 'pickup_hour', 'trip_count']].copy()
prediction_output['prediction_raw'] = test_pred_raw
prediction_output['prediction'] = test_pred
prediction_output['error'] = prediction_output['trip_count'] - prediction_output['prediction']
prediction_output['absolute_error'] = prediction_output['error'].abs()

metrics_output = model_1_metrics.copy()
metrics_output['model'] = 'Model 1 - Linear Regression'
metrics_output = metrics_output[['model', 'dataset', 'MAE', 'RMSE', 'R2', 'rows']]

MODEL_PATH = MODELING_DIR / 'model_1_linear_regression.joblib'
PREDICTIONS_PATH = MODELING_DIR / 'model_1_predictions.csv'
METRICS_PATH = MODELING_DIR / 'model_1_metrics.csv'

joblib.dump(model_1, MODEL_PATH)
prediction_output.to_csv(PREDICTIONS_PATH, index=False)
metrics_output.to_csv(METRICS_PATH, index=False)

print('Saved:')
print(' -', MODEL_PATH)
print(' -', PREDICTIONS_PATH)
print(' -', METRICS_PATH)


Saved:
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/data/modeling/model_1_linear_regression.joblib
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/data/modeling/model_1_predictions.csv
 - /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/data/modeling/model_1_metrics.csv


## 11. Inspect Linear Regression coefficients

This is useful for interpretation, but coefficient magnitude should be interpreted carefully because the demand variables are correlated.


In [11]:
fitted_preprocessor = model_1.named_steps['preprocessor']
fitted_regressor = model_1.named_steps['regressor']

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_regressor.coef_

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'absolute_coefficient': np.abs(coefficients)
}).sort_values('absolute_coefficient', ascending=False)

display(coef_df.head(25))

COEF_PATH = MODELING_DIR / 'model_1_coefficients.csv'
coef_df.to_csv(COEF_PATH, index=False)
print('Saved:', COEF_PATH)


,feature,coefficient,absolute_coefficient
123,categorical__PULocationID_237,-12.919343,12.919343
63,categorical__PULocationID_132,-11.876866,11.876866
83,categorical__PULocationID_161,-11.629459,11.629459
122,categorical__PULocationID_236,-11.004577,11.004577
94,categorical__PULocationID_186,-8.612031,8.612031
84,categorical__PULocationID_162,-8.168467,8.168467
143,categorical__hour_7,8.006840,8.006840
117,categorical__PULocationID_230,-7.609603,7.609603
153,categorical__hour_17,7.272624,7.272624
71,categorical__PULocationID_142,-7.132501,7.132501


Saved: /Users/sadhanatp/Desktop/Amrita /Semester VII/23CSE452/Case Study/Group/nyc-taxi-demand-forecasting/data/modeling/model_1_coefficients.csv


## 12. Quick residual check

This is an initial diagnostic only. Full model comparison and error analysis will be done after Models 1–3 are trained.


In [12]:
test_results = prediction_output.copy()
test_results['hour'] = test_results['pickup_hour'].dt.hour

hourly_error = (
    test_results.groupby('hour', as_index=False)
    .agg(
        MAE=('absolute_error', 'mean'),
        actual_mean=('trip_count', 'mean'),
        prediction_mean=('prediction', 'mean')
    )
)

display(hourly_error)


,hour,MAE,actual_mean,prediction_mean
0,0,6.618967,23.429028,24.764835
1,1,4.770959,15.413257,16.627872
2,2,3.451158,10.358269,11.541030
3,3,3.038672,7.245951,8.419072
4,4,3.073571,5.827122,7.130939
5,5,3.419837,6.593375,7.690878
6,6,5.311202,13.307660,14.532750
7,7,9.225290,23.934990,25.776831
8,8,7.145277,31.528157,33.339467
9,9,6.624995,32.663975,35.232893


# Model 1 complete

The saved outputs can now be used when training Model 2 and Model 3.

### Files produced
- `data/modeling/model_1_linear_regression.joblib`
- `data/modeling/model_1_predictions.csv`
- `data/modeling/model_1_metrics.csv`
- `data/modeling/model_1_coefficients.csv`

**Do not tune Model 1 yet.** First train Models 2 and 3 with the same train/test split and metrics. Then perform the model comparison and decide whether tuning is justified.
